# SignVLM: batch video augmentation (Colab)

Generates up to **9** augmented `*.mp4` files per **label folder** in `validation_data`, `test_data`, and `unseen_data` only (not `train_data`), in **place** (new files in the same class subfolders; no separate output root).

Uses `augmentation_combined.py` (OpenCV `mp4v`). A **class folder** is **skipped** if it already has any `*_aug_*.mp4`. Remaining classes are run **in parallel** (`ProcessPoolExecutor`); set **`AUG_MAX_WORKERS`** (e.g. `2` or `4`) to cap processes if you hit OOM. With 9 per label, new files are `{label}_aug_16` … `aug_24` (existing target names are skipped inside the run).

**Data on Drive:** `SIGNVLM_DRIVE_BASE` (default on Colab: `.../MyDrive/FYP_DATA`), `SIGNVLM_BASE` when not on Colab.

**Augmentation code:** **Cell 2b** clones the repo from **GitHub** into `/content/...` (same env as training: `SIGNVLM_GIT_URL`, `SIGNVLM_GIT_BRANCH`, `SIGNVLM_GIT_DIR`, `SIGNVLM_GIT_DEPTH`, `SIGNVLM_GIT_RECLONE`). If git fails, a copy under `{BASE}/Urdu-Sign-Language-Recognition-using-SignVLM` (or `SIGNVLM_REPO_ROOT`) is used when it contains `augmentation_combined.py`.

In-place: no new dataset directories; a missing split folder is skipped with one line.

In [ ]:
# Installs: OpenCV (video read/write) + NumPy; works with Python 3 on Colab and locally.
import subprocess
import sys

def pip_install(packages):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *packages])

pip_install(["opencv-python-headless", "numpy"])
print("OK: opencv-python-headless, numpy")

In [ ]:
# Mount Drive (Colab) and set paths — see markdown above for env vars.
import os
from pathlib import Path

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_MY = "/content/drive/MyDrive"
else:
    DRIVE_MY = os.environ.get("SIGNVLM_DRIVE_MY", "")

BASE = os.environ.get("SIGNVLM_DRIVE_BASE", "").strip()
if not BASE:
    if IN_COLAB and DRIVE_MY:
        BASE = f"{DRIVE_MY}/FYP_DATA"
    else:
        BASE = os.environ.get("SIGNVLM_BASE", os.path.join(os.getcwd(), "signvlm_data"))

REPO_SUBDIR = os.environ.get("SIGNVLM_REPO_SUBDIR", "Urdu-Sign-Language-Recognition-using-SignVLM").strip()
REPO_ROOT = os.environ.get("SIGNVLM_REPO_ROOT", "").strip()
if not REPO_ROOT:
    REPO_ROOT = str(Path(BASE) / REPO_SUBDIR)
REPO_ROOT = str(Path(REPO_ROOT).resolve())
AUG_DIR = Path(REPO_ROOT) / "augmentation_and_preprocessing" / "augmentation"
print(f"IN_COLAB={IN_COLAB} BASE={BASE}\nREPO_ROOT={REPO_ROOT}\nAUG_DIR={AUG_DIR}")
if not Path(BASE).is_dir():
    print("WARNING: BASE is not a directory. Set SIGNVLM_DRIVE_BASE or signvlm_data layout.")
if not AUG_DIR.is_dir():
    print(
        f"note: no augmentation tree at {AUG_DIR} (expected if you will use **Cell 2b** git clone). "
        "Run the next cell to load code from GitHub."
    )

## Cell 2b: Augmentation code from GitHub (recommended on Colab)

`BASE` (videos) can stay on **Google Drive**; the **Python** in `augmentation_combined.py` is loaded from a **shallow `git clone`** into `/content/...` so you always get the current helpers (no stale Drive copy).

**Env (optional):** `SIGNVLM_GIT_URL` (default: `ZainnB/Urdu-Sign-Language-Recognition-using-SignVLM` on GitHub), `SIGNVLM_GIT_BRANCH` (default: `training_thru_tensors`—match your branch or set e.g. `main`), `SIGNVLM_GIT_DIR` (clone under `/content/...` by default), `SIGNVLM_GIT_DEPTH`, `SIGNVLM_GIT_RECLONE=1` to re-clone. If git fails and a valid copy exists under `BASE/.../Urdu-Sign-Language-Recognition-using-SignVLM` on Drive, that path is used.

**Private repo:** set `os.environ['SIGNVLM_GIT_URL']` to `https://<token>@github.com/.../repo.git` or use Colab **Secrets** for the token in the URL.

In [ ]:
# Clone / update repo from GitHub; set REPO_ROOT + AUG_DIR to that tree (keep BASE for videos on Drive from cell 2).
import os
import shutil
import subprocess
from pathlib import Path

try:
    import google.colab  # noqa: F401
    _IN_COLAB = True
except ImportError:
    _IN_COLAB = False

WORK_ROOT = "/content" if _IN_COLAB else os.getcwd()
DRIVE_REPO_ROOT = str(Path(BASE) / REPO_SUBDIR)

def _has_augmentation_py(path: str) -> bool:
    p = Path(path) / "augmentation_and_preprocessing" / "augmentation" / "augmentation_combined.py"
    return p.is_file()

DEFAULT_GIT_URL = "https://github.com/ZainnB/Urdu-Sign-Language-Recognition-using-SignVLM.git"
DEFAULT_GIT_BRANCH = "training_thru_tensors"
GIT_URL = os.environ.get("SIGNVLM_GIT_URL", DEFAULT_GIT_URL).strip()
GIT_BRANCH = (os.environ.get("SIGNVLM_GIT_BRANCH", DEFAULT_GIT_BRANCH) or DEFAULT_GIT_BRANCH).strip()
try:
    GIT_DEPTH = int(os.environ.get("SIGNVLM_GIT_DEPTH", "1") or "1")
except ValueError:
    GIT_DEPTH = 1
GIT_DIR = os.environ.get("SIGNVLM_GIT_DIR", "").strip() or os.path.join(
    WORK_ROOT, "Urdu-Sign-Language-Recognition-using-SignVLM"
)
RECLONE = os.environ.get("SIGNVLM_GIT_RECLONE", "").strip().lower() in ("1", "true", "yes")

REPO_RESOLVED = None
if GIT_URL:
    if RECLONE and os.path.isdir(GIT_DIR):
        shutil.rmtree(GIT_DIR, ignore_errors=True)
    try:
        if os.path.isdir(os.path.join(GIT_DIR, ".git")):
            subprocess.run(
                ["git", "-C", GIT_DIR, "fetch", "--all", "--prune"],
                check=False,
            )
            subprocess.run(["git", "-C", GIT_DIR, "checkout", GIT_BRANCH], check=True)
            subprocess.run(
                ["git", "-C", GIT_DIR, "pull", "--ff-only"],
                check=False,
            )
        else:
            if os.path.exists(GIT_DIR):
                shutil.rmtree(GIT_DIR, ignore_errors=True)
            parent = os.path.dirname(GIT_DIR) or "."
            os.makedirs(parent, exist_ok=True)
            cmd = ["git", "clone"]
            if GIT_DEPTH > 0:
                cmd.extend(["--depth", str(GIT_DEPTH)])
            cmd.extend(["-b", GIT_BRANCH, GIT_URL, GIT_DIR])
            print("Running:", " ".join(cmd))
            subprocess.run(cmd, check=True)
    except (subprocess.CalledProcessError, OSError) as e:
        print("Git error (trying Drive if available):", e)
    if _has_augmentation_py(GIT_DIR):
        REPO_RESOLVED = str(Path(GIT_DIR).resolve())
        print("Using SignVLM augmentation from git:", REPO_RESOLVED, "branch:", GIT_BRANCH)

if not REPO_RESOLVED and _has_augmentation_py(DRIVE_REPO_ROOT):
    REPO_RESOLVED = str(Path(DRIVE_REPO_ROOT).resolve())
    print("Using SignVLM augmentation from Drive:", REPO_RESOLVED)

if not REPO_RESOLVED:
    raise FileNotFoundError(
        f"No augmentation_combined.py. Set SIGNVLM_GIT_URL, run git successfully, or place a repo with "
        f"augmentation_and_preprocessing/augmentation/ under: {DRIVE_REPO_ROOT} (or export SIGNVLM_GIT_DIR)."
    )

REPO_ROOT = REPO_RESOLVED
AUG_DIR = Path(REPO_ROOT) / "augmentation_and_preprocessing" / "augmentation"
print(f"REPO_ROOT={REPO_ROOT}\nAUG_DIR={AUG_DIR}")
if not (AUG_DIR / "augmentation_combined.py").is_file():
    raise FileNotFoundError(f"augmentation_combined.py not found: {AUG_DIR}")

In [ ]:
# Import augmentation (adds same folder that holds augmentation_combined.py).
# If your Drive copy is old (missing helper names), we fall back to the inline defs using augment_video_random.
import os
import random
import sys
from concurrent.futures import ProcessPoolExecutor
from pathlib import Path

aug = Path(REPO_ROOT) / "augmentation_and_preprocessing" / "augmentation"
_augp = str(aug)
if _augp not in os.environ.get("PYTHONPATH", ""):
    os.environ["PYTHONPATH"] = _augp + (
        os.pathsep + os.environ["PYTHONPATH"] if os.environ.get("PYTHONPATH") else ""
    )
if _augp not in sys.path:
    sys.path.insert(0, _augp)

import augmentation_combined as _ac  # noqa: E402


def _label_dir_has_aug_videos_local(label_folder):
    return any(Path(label_folder).glob("*_aug_*.mp4"))


def _augment_one_label_local(label_folder, num_augmented_per_label, verbose=True):
    label_folder = Path(label_folder)
    label_name = label_folder.name
    original_videos = sorted(
        v for v in label_folder.glob("*.mp4") if "_aug_" not in v.name
    )
    if not original_videos:
        if verbose:
            print(f"{label_name}: ❌ No original videos found")
        return 0, 0, label_name
    if verbose:
        print(
            f"Label: {label_name} | Original videos: {len(original_videos)} | Generating: {num_augmented_per_label}"
        )
    successful = 0
    failed = 0
    for aug_idx in range(num_augmented_per_label):
        source_video = random.choice(original_videos)
        output_num = 15 + aug_idx + 1
        output_name = f"{label_name}_aug_{output_num}.mp4"
        output_path = label_folder / output_name
        if output_path.exists():
            if verbose:
                print(f"  [{aug_idx+1}/{num_augmented_per_label}] {output_name} ⏭️  (exists)", end="")
            continue
        if verbose:
            print(f"  [{aug_idx+1}/{num_augmented_per_label}] {source_video.name} → {output_name}", end="")
        try:
            success = _ac.augment_video_random(
                input_video=str(source_video), output_video=str(output_path)
            )
            if success:
                if verbose:
                    print(" ✓")
                successful += 1
            else:
                if verbose:
                    print(" ❌")
                failed += 1
                if output_path.exists():
                    output_path.unlink()
        except Exception as e:
            if verbose:
                print(f" ❌ ({str(e)[:40]})")
            failed += 1
    if verbose:
        print(f"  → {label_name}: {successful} successful, {failed} failed\n")
    return successful, failed, label_name


label_dir_has_aug_videos = getattr(
    _ac, "label_dir_has_aug_videos", _label_dir_has_aug_videos_local
)
augment_one_label = getattr(_ac, "augment_one_label", _augment_one_label_local)
if getattr(_ac, "augment_one_label_entry", None) is not None:
    augment_one_label_entry = _ac.augment_one_label_entry
else:
    def augment_one_label_entry(packed):
        path, n, verbose = packed
        return augment_one_label(path, n, verbose=verbose)

if (not hasattr(_ac, "augment_one_label")) or (not hasattr(_ac, "augment_one_label_entry")):
    print("Note: in-notebook fallbacks (git clone in Cell 2b should provide the full module).")

In [ ]:
# Per-label: skip if the class folder already has any *_aug_*.mp4; else augment in parallel.
from pathlib import Path

SPLITS = ("validation_data", "test_data", "unseen_data")
AUGMENTED_PER_LABEL = 9
MAX_WORKERS = max(1, int(os.environ.get("AUG_MAX_WORKERS", "4")))


def collect_jobs():
    jobs = []
    skipped_aug = []
    for split_name in SPLITS:
        root = Path(BASE) / split_name
        if not root.is_dir():
            print(f"skip (not found): {root}")
            continue
        for label_dir in sorted(d for d in root.iterdir() if d.is_dir()):
            if label_dir_has_aug_videos(label_dir):
                skipped_aug.append(f"{split_name}/{label_dir.name}")
                continue
            jobs.append((str(label_dir), AUGMENTED_PER_LABEL, True))
    return jobs, skipped_aug


jobs, skipped_aug = collect_jobs()
if skipped_aug:
    print(f"Skipped {len(skipped_aug)} label(s) (already have *_aug_*.mp4):")
    for s in skipped_aug:
        print("  -", s)
if not jobs:
    print("Nothing to do: all present labels have aug, or no label folders found.")
else:
    n_workers = min(MAX_WORKERS, len(jobs))
    print(
        f"Running {len(jobs)} label(s) with {n_workers} process(es) "
        "(set AUG_MAX_WORKERS; worker logs may interleave)."
    )
    total_s, total_f = 0, 0
    with ProcessPoolExecutor(max_workers=n_workers) as ex:
        for s, f, _ in ex.map(augment_one_label_entry, jobs, chunksize=1):
            total_s += s
            total_f += f
    print(f"Done. total success: {total_s}, total failed: {total_f}")